# Veltis — NumPy cosine retrieval: 100 vs 544 tokens

**Adapté de `ex12-10-26_chunking_embeddings-2.ipynb`.** Aucun Qdrant. Original conservé sans modification.

Objectif : reproduire les recherches Q01–Q03 sur le **guide seul** et les **13 documents** avec le nouveau chunker **100 tokens**, puis comparer à la configuration 544 sauvegardée hier.

**Dépendance :** BGE-M3 en service sur `127.0.0.1:8001` pour calculer les nouveaux embeddings et les questions. L'exécution du notebook nécessite la même installation Python que l'original.

**Attention :** `max_tokens=100` est une cible de découpage ; mesurer la longueur réelle des passages contextualisés. Les métriques basées sur le texte exact doivent être vérifiées manuellement si la réponse est divisée entre plusieurs chunks.

In [1]:
# CELL 1 — ENVIRONMENT AND SOURCE DOCUMENTS
from pathlib import Path
import json
import numpy as np
from docling_core.types.doc import DoclingDocument
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
from openai import OpenAI

PROJECT_ROOT = Path('/home/bm/formation/RAG/cours-rag/drive-download-20261008T064617Z-1-001/rag/demo')
EXPORT_FOLDER = PROJECT_ROOT / 'export_folder'
MODEL_PATH = Path('/home/bm/models/rag/embedder/bge-m3')

# Sélection explicite des exports Docling; pas des JSON de métadonnées RAG.
json_files = sorted(f for f in EXPORT_FOLDER.glob('*.json') if f.name.endswith(('.pdf.json', '.html.json', '.docx.json', '.pptx.json')))
assert len(json_files) == 13, f'13 exports attendus, {len(json_files)} trouvés: {[f.name for f in json_files]}'
assert MODEL_PATH.is_dir()
assert (EXPORT_FOLDER / 'veltis_embeddings.npz').is_file()
assert (EXPORT_FOLDER / 'veltis_chunks.json').is_file()
print('Documents Docling:', len(json_files))
print('Corpus original 544 disponible: oui')

Documents Docling: 13
Corpus original 544 disponible: oui


In [2]:
# CELL 2 — LOAD TOKENIZER AND CREATE 100-TOKEN CHUNKER
TOKEN_TARGET_100 = 100
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
chunker_100 = HybridChunker(tokenizer=HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=TOKEN_TARGET_100))
print('Tokenizer:', tokenizer.__class__.__name__)
print('Chunker target:', TOKEN_TARGET_100)

Tokenizer: XLMRobertaTokenizer
Chunker target: 100


In [3]:
# CELL 3 — CHUNK THE 13 DOCUMENTS (100-TOKEN CONFIGURATION)
chunks_100 = []
for file in json_files:
    doc = DoclingDocument.load_from_json(file)
    parts = list(chunker_100.chunk(doc))
    for part in parts:
        chunks_100.append({'source': file.name.removesuffix('.json'), 'text': chunker_100.contextualize(part)})
    print(f'{file.name}: {len(parts)} chunks')
texts_100 = [x['text'] for x in chunks_100]
tokens_100 = [len(tokenizer.encode(t)) for t in texts_100]
assert texts_100 and all(t.strip() for t in texts_100)
print('\nTotal 100-profile chunks:', len(chunks_100))
print('Actual token sizes: min', min(tokens_100), '| max', max(tokens_100))
print('Above target 100:', sum(t > TOKEN_TARGET_100 for t in tokens_100))
print('Above 128:', sum(t > 128 for t in tokens_100))

grille_salariale_2026.pdf.json: 12 chunks
guide_onboarding.pdf.json: 14 chunks
intranet_actualites.html.json: 5 chunks
intranet_faq_rh.html.json: 8 chunks
newsletter_le_fil_2026-02.pdf.json: 11 chunks
organigramme_et_processus_production.pdf.json: 2 chunks
plan_strategique_2026-2028.pptx.json: 5 chunks
politique_teletravail_2023.docx.json: 7 chunks
politique_teletravail_2025.docx.json: 8 chunks
politique_teletravail_2025.pdf.json: 8 chunks
presentation_veltis.pdf.json: 11 chunks
pv_conseil_administration_2026-03-12_scan.pdf.json: 8 chunks
rapport_annuel_2025.pdf.json: 30 chunks

Total 100-profile chunks: 129
Actual token sizes: min 9 | max 110
Above target 100: 31
Above 128: 0


Les textes de plus de 100 tokens ne sont **pas** tronqués ici. Noter leurs longueurs et inspecter les passages inhabituels avant de les indexer ; BGE-M3 accepte les passages de cet exercice. Le test compare une **configuration 100** à une **configuration 544**, non une garantie de longueur exacte.

In [4]:
# CELL 4 — RELOAD YESTERDAY'S 544-TOKEN DATA (NO RECOMPUTATION)
with np.load(EXPORT_FOLDER / 'veltis_embeddings.npz') as data:
    embeddings_544 = data['embeddings'].astype(np.float32)
with open(EXPORT_FOLDER / 'veltis_chunks.json', encoding='utf-8') as f:
    chunks_544 = json.load(f)
assert embeddings_544.shape == (84, 1024)
assert len(chunks_544) == embeddings_544.shape[0]
assert np.isfinite(embeddings_544).all()
norms_544 = np.linalg.norm(embeddings_544, axis=1)
assert np.all(norms_544 > 0)
normalized_544 = embeddings_544 / norms_544[:, None]
print('544: chunks', len(chunks_544), '| matrix', embeddings_544.shape)

544: chunks 84 | matrix (84, 1024)


In [5]:
# CELL 5 — CONNECT TO THE SAME LOCAL BGE-M3 SERVER USED YESTERDAY
embedding_client = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='not-needed', timeout=120.0)
print([model.id for model in embedding_client.models.list().data])
assert any(model.id == 'bge-m3' for model in embedding_client.models.list().data)

['bge-m3']


In [6]:
# CELL 6 — GENERATE 100-PROFILE EMBEDDINGS IN SMALL BATCHES
# Conserver le même ordre entre chunks_100 et les vecteurs.
BATCH_SIZE = 16
vectors_100 = []
for start in range(0, len(texts_100), BATCH_SIZE):
    batch = texts_100[start:start + BATCH_SIZE]
    reply = embedding_client.embeddings.create(model='bge-m3', input=batch)
    ordered = sorted(reply.data, key=lambda item: item.index)
    assert len(ordered) == len(batch)
    vectors_100.extend(item.embedding for item in ordered)
    print(f'Embedded {len(vectors_100)}/{len(texts_100)}')
embeddings_100 = np.asarray(vectors_100, dtype=np.float32)
assert embeddings_100.shape == (len(chunks_100), 1024)
assert np.isfinite(embeddings_100).all()
norms_100 = np.linalg.norm(embeddings_100, axis=1)
assert np.all(norms_100 > 0)
normalized_100 = embeddings_100 / norms_100[:, None]
print('100: matrix', embeddings_100.shape)

Embedded 16/129
Embedded 32/129
Embedded 48/129
Embedded 64/129
Embedded 80/129
Embedded 96/129
Embedded 112/129
Embedded 128/129
Embedded 129/129
100: matrix (129, 1024)


In [7]:
# CELL 7 — SAVE 100-PROFILE EMBEDDINGS IN DIFFERENT FILES
path_vectors_100 = EXPORT_FOLDER / 'veltis_embeddings_100.npz'
path_chunks_100 = EXPORT_FOLDER / 'veltis_chunks_100.json'
np.savez_compressed(path_vectors_100, embeddings=embeddings_100)
with open(path_chunks_100, 'w', encoding='utf-8') as f:
    json.dump(chunks_100, f, ensure_ascii=False, indent=2)
print('Saved:', path_vectors_100.name, '|', path_chunks_100.name)
print('Original 544 files unchanged.')

Saved: veltis_embeddings_100.npz | veltis_chunks_100.json
Original 544 files unchanged.


## Recherche cosinus — quatre scénarios

- 544 / **guide seul** : filtrer les chunks de `guide_onboarding.pdf` avant le calcul du Top 5.
- 544 / **13 documents** : tous les chunks.
- 100 / **guide seul** : même filtre avant le Top 5.
- 100 / **13 documents** : tous les chunks.

Les **mêmes embeddings de question** sont employés pour les quatre scénarios. La méthode est celle du notebook initial : normalisation L2 puis produit matriciel NumPy (`@`).

In [8]:
# CELL 8 — COSINE SEARCH WITH OPTIONAL SOURCE FILTER
GUIDE = 'guide_onboarding.pdf'
def cosine_search(question_vector, records, matrix, top_k=5, source=None):
    allowed = [i for i, r in enumerate(records) if source is None or r['source'] == source]
    if not allowed:
        raise ValueError(f'Aucun chunk dans la source {source}')
    v = np.asarray(question_vector, dtype=np.float32)
    length = np.linalg.norm(v)
    if length == 0:
        raise ValueError('Question vector has zero norm')
    # matrix already L2-normalized, normalize the question too.
    scores = matrix[allowed] @ (v / length)
    ranked_positions = np.argsort(scores)[::-1][:top_k]
    return [dict(global_index=allowed[j], source=records[allowed[j]]['source'], score=float(scores[j]), text=records[allowed[j]]['text']) for j in ranked_positions]
print('Ready: NumPy cosine retrieval; no Qdrant.')

Ready: NumPy cosine retrieval; no Qdrant.


In [9]:
# CELL 9 — REUSE THE SAME THREE QUESTIONS; COMPUTE EMBEDDINGS ONCE
questions = {
    'Q01': 'Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?',
    'Q02': 'Pendant combien de temps vais-je être accompagné par un collègue à mon arrivée ?',
    'Q03': 'Que se passe-t-il si je ne change pas mon mot de passe provisoire ?',
}
expected = {
    'Q01': 'Présentez-vous à 8 h 30',
    'Q02': 'pendant ses 3 mois premiers mois',
    'Q03': 'le compte est bloqué',
}
reply = embedding_client.embeddings.create(model='bge-m3', input=list(questions.values()))
ordered = sorted(reply.data, key=lambda item: item.index)
question_vectors = {qid: np.array(item.embedding, dtype=np.float32) for qid, item in zip(questions, ordered)}
assert all(v.shape == (1024,) for v in question_vectors.values())
print('Question embeddings generated:', list(question_vectors))

Question embeddings generated: ['Q01', 'Q02', 'Q03']


In [10]:
# CELL 10 — EVALUATE Q01–Q03: 544 VS 100, GUIDE VS ALL 13 DOCS
scenarios = {
    '544 / guide': (chunks_544, normalized_544, GUIDE),
    '544 / tous': (chunks_544, normalized_544, None),
    '100 / guide': (chunks_100, normalized_100, GUIDE),
    '100 / tous': (chunks_100, normalized_100, None),
}
measurements = {}
for scenario, (records, matrix, source_filter) in scenarios.items():
    ranks = {}
    print(f'\n===== {scenario.upper()} =====')
    print('Candidates:', sum(r['source'] == GUIDE for r in records) if source_filter else len(records))
    for qid, question in questions.items():
        found = cosine_search(question_vectors[qid], records, matrix, top_k=5, source=source_filter)
        rank_correct = None
        print(f'\n{qid}: {question}')
        for rank, hit in enumerate(found, 1):
            is_correct = hit['source'] == GUIDE and expected[qid].casefold() in hit['text'].casefold()
            if is_correct and rank_correct is None:
                rank_correct = rank
            label = ' << RÉPONSE ATTENDUE >>' if is_correct else ''
            print(f'  {rank}. {hit["source"]} | global={hit["global_index"]} | cosine={hit["score"]:.4f}{label}')
            print('    ', hit['text'][:350].replace('\n',' '))
        ranks[qid] = rank_correct
        print('Rang de la réponse exacte:', rank_correct if rank_correct else 'absente du Top 5')
    measurements[scenario] = ranks
print('\n===== TABLEAU COMPARATIF =====')
print(f'{"Scénario":17} | Q01 | Q02 | Q03 | Hit@1 | Hit@5 | MRR@5')
for scenario, ranks in measurements.items():
    rr = [ranks[k] for k in questions]
    hit1 = sum(r == 1 for r in rr)
    hit5 = sum(r is not None for r in rr)
    mrr = sum((1 / r) if r else 0 for r in rr) / len(rr)
    fmt = lambda r: str(r) if r else '—'
    print(f'{scenario:17} | {fmt(rr[0]):>3} | {fmt(rr[1]):>3} | {fmt(rr[2]):>3} | {hit1:>5}/3 | {hit5:>5}/3 | {mrr:.3f}')
print('\nNote: réponse exacte = la chaîne attendue apparaît dans un seul chunk du guide. Vérifier manuellement les faux négatifs dus au découpage.')


===== 544 / GUIDE =====
Candidates: 13

Q01: Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?
  1. guide_onboarding.pdf | global=5 | cosine=0.6002
     1. Bienvenue chez Veltis Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines
  2. guide_onboarding.pdf | global=10 | cosine=0.5838
     3.3 Accès informatiques Votre compte est créé par le service IT avant votre arrivée. Le mot de passe provisoire doit être changé dans les 24 heures, faute de quoi le compte est bloqué. En cas de problème, appelez le helpdesk au poste 2100 ou écrivez à helpdesk@veltis.example.
  3. guide_onboarding.pdf | global=8 | cosine=0.5547 << RÉPONSE ATTENDUE >>
     3.1 Accueil Présentez-vous à 8 h 30 à l'accueil du bâtiment A. Votre badge d'accès vous y sera remis. Il ouvre les bâtiments, le 

### Interprétation

- **Rang plus petit = meilleur**, pour la question considérée.
- **Hit@1** : présence de la preuve attendue en première position.
- **Hit@5** : présence dans les cinq premiers.
- **MRR@5** : moyenne de `1/rang` si le passage est dans le Top 5, sinon 0.
- La recherche **guide seul** est filtrée **avant** le classement ; il ne s'agit pas de filtrer les cinq résultats du corpus complet.
- Le test utilise une phrase exacte par question ; si les chunks de 100 tokens divisent la preuve, le rang automatique peut être `—` même si le passage est pertinent. Examiner le texte affiché.
- Comparer aussi au **résultat observé hier pour 544 / tous** : Q01=5, Q02=1, Q03=1, MRR@5=0,733. Des écarts légers de scores sont possibles.
- Les données 544 proviennent du fichier sauvegardé ; aucune des anciennes sauvegardes n'est modifiée.